# Glow25: retention as a system

Order: **real** (PUBLIC DATA) -> **simulation** (SIMULATED, fictional "Brand G") -> cohorts -> flows -> subscriptions -> traps -> decisions.
Run the pipeline first (README, "How to re-run"); this notebook reads its outputs and a few tables of the DuckDB file.
Simulated numbers are never Glow25's.

In [1]:
import json
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PUB, PROC = ROOT / "data" / "public", ROOT / "data" / "processed"
pd.set_option("display.width", 140, "display.max_columns", 12, "display.precision", 3)
con = duckdb.connect(str(ROOT / "data" / "sim" / "retention.duckdb"), read_only=True)
f = json.loads((PROC / "headline_facts.json").read_text(encoding="utf-8"))

## 1. Real: the offer, the markets, the season, the claim rule (PUBLIC DATA)

In [2]:
cat = pd.read_csv(PUB / "glow25_catalogue.csv")
print(len(cat), "products,", int(cat.subscribable.sum()), "subscribable")
pd.read_csv(PUB / "glow25_selling_plans.csv").groupby(["plan_name", "adjustment_value"]).size()

29 products, 8 subscribable


plan_name                adjustment_value
Lieferung alle 3 Monate  25                  8
Monatliche Lieferung     15                  8
dtype: int64

In [3]:
cat[cat.handle.str.contains("probierpaket")][["handle", "price_eur", "subscribable", "portions_stated"]]

,handle,price_eur,subscribable,portions_stated
9,kollagen-plus-probierpaket-225g,19.99,False,15.0
10,kollagen-original-probierpaket-225g,14.99,False,15.0
11,kollagen-plus-strawberry-probierpaket-225g,19.99,False,15.0


In [4]:
e = pd.read_csv(PUB / "eurostat_isoc_ec_ibgs.csv")
e[e.ind_type == "F_Y25_54"].pivot_table(index="geo", columns="indic_is", values="value").sort_values("I_BMED1")

indic_is,I_BCBW,I_BMED1
geo,,
FR,29.14,11.77
IT,21.05,14.29
ES,34.68,17.42
BE,33.86,28.56
DE,34.35,28.56
NL,50.89,31.12
AT,33.17,32.09


In [5]:
pd.Series(f["public"]["wiki_jan_over_jul"], name="January / July pageviews")

de.wikipedia    1.45
es.wikipedia    1.34
fr.wikipedia    2.16
it.wikipedia    2.03
nl.wikipedia    2.07
Name: January / July pageviews, dtype: float64

In [6]:
pd.read_csv(PUB / "eu_432_2012_collagen_claims.csv").query("language == 'EN'").authorised_claim.tolist()

['Vitamin C contributes to normal collagen formation for the normal function of blood vessels',
 'Vitamin C contributes to normal collagen formation for the normal function of bones',
 'Vitamin C contributes to normal collagen formation for the normal function of cartilage',
 'Vitamin C contributes to normal collagen formation for the normal function of gums',
 'Vitamin C contributes to normal collagen formation for the normal function of skin',
 'Vitamin C contributes to normal collagen formation for the normal function of teeth']

## 2. Simulation: what was generated (SIMULATED)

In [7]:
con.sql("""select (select count(*) from dim_customer) as customers, (select count(*) from fct_order) as orders,
       (select count(*) from fct_subscription) as subscriptions, (select count(*) from email_events) as email_events""").df()

,customers,orders,subscriptions,email_events
0,120000,348294,26872,2333969


In [8]:
pd.read_csv(PROC / "data_quality_summary.csv")

,check_name,value
0,duplicate profiles (same email_hash),2300.000
1,orders sitting on a duplicate profile,8920.000
2,subscription orders whose UTC month differs fr...,3501.000
3,share of subscription orders that slip month,0.025
4,"monthly subscription churn, skipped counted as...",0.119
5,"monthly subscription churn, cancellations only...",0.074


## 3. Cohorts and CM-CLV

In [9]:
pd.read_csv(PROC / "kpi_repurchase.csv").query("segment == 'all'")

,segment,n_days,customers,repurchase_rate
0,all,30,115229,0.220
1,all,60,110761,0.379
2,all,90,106665,0.436
3,all,180,92942,0.481
4,all,365,55377,0.541


In [10]:
pd.read_csv(PROC / "cohort_cmclv_by_segment.csv").query("dimension == 'first order type'")

,dimension,segment,customers,cm_clv_3m,cm_clv_6m,cm_clv_12m,returned_within_12m
5,first order type,subscription,5945,48.939,86.300,148.521,0.838
6,first order type,one-off supplement,24735,31.984,41.055,55.181,0.499
7,first order type,trial pack / bundle,20690,23.691,33.499,50.277,0.470
8,first order type,cosmetic,4125,24.355,30.018,40.335,0.397


In [11]:
pd.read_csv(PROC / "clv_uci_backtest.csv")[["method", "revenue_jul_nov_2011_gbp", "error_vs_actual"]]

,method,revenue_jul_nov_2011_gbp,error_vs_actual
0,"BG/NBD + Gamma-Gamma (fitted, scipy)",3.106e+06,-0.181
1,Empirical chain ladder (cohort averages),2.951e+06,-0.222
2,Actual (what happened),3.791e+06,0.000


## 4. Flows: attributed vs incremental

In [12]:
pd.read_csv(PROC / "flow_holdout_readout.csv")[["flow", "n_t", "n_h", "rpr_diff", "rpr_diff_ci95", "incremental_revenue_eur",
                                               "attributed_revenue_eur", "incremental_share_of_attributed"]]

,flow,n_t,n_h,rpr_diff,rpr_diff_ci95,incremental_revenue_eur,attributed_revenue_eur,incremental_share_of_attributed
0,F2_education,74398,8205,0.853,0.472,63463.088,292849.63,0.217
1,F4_replenishment,95643,10885,1.294,0.433,123782.490,167131.86,0.741
2,F3_trial_to_sub,28087,3075,1.932,0.722,54268.333,49057.09,1.106
3,F6_cancel_save,10821,1096,6.832,1.126,73929.388,12467.71,5.930
4,F7_winback,63295,7058,-0.082,0.189,-5187.833,9484.83,-0.547
5,F1_welcome,86859,9524,-0.085,0.173,-7424.060,4753.54,-1.562
6,VIP early access (Gold),1089,125,0.448,2.982,488.308,1835.55,0.266
7,F8_vip_loyalty,1868,213,-2.013,2.846,-3761.131,NaN,NaN
8,F5_sub_onboarding,21073,2279,1.327,2.337,27969.419,NaN,NaN


In [13]:
pd.read_csv(PROC / "replenishment_j21_vs_j26.csv")

,arm,triggers,repurchase_60d,repurchase_ci95,revenue_per_recipient_60d,cm_per_recipient_60d,click_rate
0,J21,11694,0.515,0.009,20.031,11.593,0.061
1,J26_branch,11541,0.561,0.009,21.695,12.526,0.109
2,holdout (no reminder),2712,0.496,0.019,19.416,11.082,NaN


In [14]:
pd.read_csv(PROC / "lapsed_reactivation_by_month.csv").iloc[2:6]

,month,lapsed_customers,reorder_rate
2,2025-11-01,32815,0.031
3,2025-12-01,35359,0.028
4,2026-01-01,38293,0.037
5,2026-02-01,41059,0.027


In [15]:
pd.read_csv(PROC / "winback_january_readout.csv").head(3)

,entry_month,treated,holdout,conv_treated,conv_holdout,diff_pts,diff_ci95
0,2026-01-01,26943,2987,0.037,0.041,-3.357e-03,0.007
1,2026-02-01,4427,502,0.045,0.046,-4.135e-04,0.019
2,2026-03-01,4088,446,0.042,0.063,-2.071e-02,0.023


## 5. Subscriptions

In [16]:
pd.read_csv(PROC / "sub_plan_economics.csv")

,interval,subscriptions,retention_12m,cm_clv_12m_eur,revenue_12m_eur,paid_charges_12m,cm_per_paid_charge,skip_rate_per_scheduled_charge,swaps_per_sub
0,1M,7214,0.404,157.643,278.311,7.358,20.097,0.033,0.068
1,3M,3891,0.587,148.080,265.400,3.200,45.868,0.150,0.057


In [17]:
pd.read_csv(PROC / "cancel_save_readout.csv")

,arm,cancel_intents,save_rate,save_rate_ci95
0,holdout (plain cancel page),1243,0.076,0.015
1,skip / swap / pause offered,12314,0.296,0.008


In [18]:
pd.read_csv(PROC / "price_increase_too_expensive.csv")

,market,period,too_expensive_cancellations,active_sub_months,per_1000_active
0,DE,Oct 2025 - Mar 2026 (before),473.0,25750.0,18.369
1,DE,Apr - Sep 2026 (after +10 %),689.0,36775.0,18.736
2,FR,Oct 2025 - Mar 2026 (before),47.0,2731.0,17.210
3,FR,Apr - Sep 2026 (after +10 %),161.0,3865.0,41.656
4,IT,Oct 2025 - Mar 2026 (before),57.0,2215.0,25.734
5,IT,Apr - Sep 2026 (after +10 %),127.0,3104.0,40.915
6,other,Oct 2025 - Mar 2026 (before),387.0,21779.0,17.769
7,other,Apr - Sep 2026 (after +10 %),539.0,31266.0,17.239


In [19]:
pd.read_csv(PROC / "winback30_readout.csv")

,arm,recipients,return_rate,revenue_per_recipient_180d,cm_per_recipient_180d,cm_ci95
0,-30 % offer,1802,0.180,17.144,8.140,0.995
1,holdout (no offer),205,0.127,18.687,10.093,3.635


## 6. Traps: loyalty self-selection, machine opens

In [20]:
pd.read_csv(PROC / "loyalty_naive_vs_matched.csv")

,comparison,r_member,r_non,relative_lift,members_used
0,naive: members vs everyone else,0.709,0.499,0.419,NaN
1,"matched: same cohort quarter, market and numbe...",0.693,0.628,0.103,22536.0


In [21]:
pd.read_csv(PROC / "subject_line_test_readout.csv")

,variant,delivered,open_rate,open_rate_excl_machine,click_rate,click_rate_ci95,revenue_per_recipient_5d,rpr_ci95
0,A,21427,0.593,0.396,0.036,0.002,1.030,0.092
1,B,21646,0.629,0.453,0.030,0.002,1.168,0.103


## 7. Decisions: planted events vs what the SQL found

In [22]:
pd.read_csv(PROC / "event_scorecard.csv")[["event_id", "event", "found_by_sql", "decision"]]

,event_id,event,found_by_sql,decision
0,1,Attributed vs incremental revenue,"education 22% (+/- 12%), replenishment 74% (+/...","Yes: invest in replenishment, not in more educ..."
1,2,Replenishment reminder too early,+4.6 pts (+/- 1.3),Yes: roll out day 26 + branch
2,3,January false signal (do NOT scale),lapsed reorders +31% Jan vs Dec; flow 3.7% vs ...,"Yes: not scaled, re-test in March"
3,4,Every-3-months plan: retention vs margin,retention 59% vs 40%; CM-CLV -6%,Yes: lead with skip / swap on the monthly plan...
4,5,Price increase and -30 % win-back,"FR x2.4, IT x1.6, DE x1.0; return 18% vs 13%, ...","Yes: stop the -30 % offer, test an upgrade pack"
5,6,Loyalty self-selection,"naive +42%, matched +10%","Yes: programme kept, its value restated"
6,7,Apple MPP opens,open rate 43% -> 61% (30% machine); B opens 62...,Yes: tests judged on clicks and revenue per re...
7,8,Data issues,"2,300 duplicates; churn 7.4% vs naive 11.9%; 2...","Yes: fixed in 01_staging.sql, tested in 99_tes..."


**Decisions**
1. Report incremental contribution margin per flow (permanent 10 % holdout) next to attributed revenue.
2. Replenishment reminder on day 26 with a "still have some?" branch.
3. Bridge the 225 g trial pack to the subscription.
4. Save subscribers with skip / swap / pause; stop the -30 % comeback offer; test an upgrade pack.
5. Localise: FR flows lead with the beauty ritual; claims from the authorised list only.